# Preprocesamiento y transformación de datos — Ejemplos

**Módulo 2 — Transformación y visualización de datos · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`2.1_Preprocesamiento_transformacion_datos.pdf`](2.1_Preprocesamiento_transformacion_datos.pdf) y lleva a código lo que allí se explica, usando un dataset real y muy conocido: el **Titanic** de Kaggle.

## Contenido

1. **Cargar el dataset de Kaggle**: el *Titanic Dataset* — pasajeros del Titanic, con su edad, clase, tarifa pagada y si sobrevivieron o no.
2. **Diagnóstico de datos faltantes**: qué columnas tienen huecos y qué tan graves son.
3. **División train/test *antes* de tratar los datos**, para evitar fuga de información (data leakage).
4. **Eliminar vs. imputar**: decisiones justificadas para cada columna con faltantes (incluyendo el efecto de valores extremos en media vs. mediana).
5. **Discretización** de la edad: por ancho uniforme, por frecuencia uniforme y de forma supervisada (usando la entropía).
6. **Codificación de variables categóricas**: binaria, one-hot y dummy.
7. **Actividad para discutir en clase**, siguiendo la estructura de la diapositiva final.

> 💡 La sección 1 **requiere una cuenta de Kaggle y una clave de API** (gratis). Si ya la configuraste para el notebook `1.6` del Módulo 1, no necesitas hacer nada más. Si no, en la sección 1 se explica paso a paso.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 5)

print('Librerías cargadas correctamente ✅')
print('pandas', pd.__version__)

---
## 1. Cargar el dataset de Kaggle: *Titanic Dataset*

El [**Titanic Dataset**](https://www.kaggle.com/datasets/yasserh/titanic-dataset) es probablemente el dataset más usado del mundo para enseñar aprendizaje automático: contiene a los pasajeros del Titanic (edad, sexo, clase, tarifa, puerto de embarque...) y si cada uno sobrevivió o no al naufragio. Tiene justo lo que necesitamos para esta unidad: **datos faltantes reales**, variables numéricas y categóricas, y una variable objetivo (`Survived`) para la discretización supervisada.

### Cómo obtener tu clave de API de Kaggle (si no la tienes de un notebook anterior)

1. Crea una cuenta gratuita en [kaggle.com](https://www.kaggle.com/) si no tienes una.
2. Ve a tu perfil → **Settings** → sección **API** → botón **"Create New Token"**. Esto descarga un archivo `kaggle.json` con tus credenciales.
3. Instala la librería oficial de Kaggle (si no la tienes): `pip install kagglehub`
4. La primera vez que ejecutes la celda de descarga, `kagglehub` te pedirá autenticarte (puede abrir el navegador, o puedes colocar el archivo `kaggle.json` descargado en `~/.kaggle/kaggle.json`).

> Tu clave de API es personal — trátala como una contraseña y no la subas al repositorio.

In [ ]:
import kagglehub
import os

try:
    ruta_dataset = kagglehub.dataset_download('yasserh/titanic-dataset')
    print('Dataset descargado en:', ruta_dataset)
except Exception as error:
    print('❌ No se pudo descargar el dataset de Kaggle.')
    print('Verifica tu conexión a internet y que tu API key esté configurada (ver celda anterior).')
    print('Detalle del error:', error)
    raise

# Buscamos el primer archivo .csv en la carpeta descargada, en vez de asumir
# un nombre exacto (así el código no se rompe si Kaggle cambia el nombre del archivo).
archivos_csv = [f for f in os.listdir(ruta_dataset) if f.endswith('.csv')]
print('Archivos CSV encontrados:', archivos_csv)

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])
titanic = pd.read_csv(ruta_csv)

print(f'\nDataset cargado: {titanic.shape[0]} filas x {titanic.shape[1]} columnas')
titanic.head()

In [ ]:
titanic.info()

---
## 2. Diagnóstico de datos faltantes

Igual que en la diapositiva 2: calculamos **qué porcentaje de cada columna está vacío**, para decidir después qué hacer con cada una.

In [ ]:
faltantes = (titanic.isnull().mean() * 100).sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]

print('Columnas con datos faltantes (%):')
print(faltantes.round(1))

fig, ax = plt.subplots()
ax.barh(faltantes.index, faltantes.values, color='#2e7d32')
ax.set_xlabel('% de datos faltantes')
ax.set_title('Diagnóstico de datos faltantes — Titanic')
ax.invert_yaxis()  # para que la columna con más faltantes quede arriba
plt.tight_layout()
plt.show()

**Lectura del diagnóstico** (como en la diapositiva 2, esto orienta la decisión de tratamiento):

- **`Cabin`** (~77% faltante): casi toda la columna está vacía. Con tan poca información disponible, **excluirla** es razonable (diapositiva 3).
- **`Age`** (~20% faltante): faltan bastantes valores, pero es una variable demasiado relevante para descartarla — mejor **imputarla**.
- **`Embarked`** (~0.2% faltante, solo 2 filas): tan pocos casos que **imputar con la moda** es más simple que eliminar filas.

---
## 3. Prevención de la fuga de información: dividir *antes* de tratar

Como se explica en la diapositiva 5, **las medias, medianas y categorías que usemos para imputar deben calcularse solo con los datos de entrenamiento**, nunca con el dataset completo — si no, estaríamos filtrando información del conjunto de prueba hacia el entrenamiento.

Por eso dividimos **antes** de calcular cualquier estadístico de imputación.

In [ ]:
train, test = train_test_split(
    titanic, test_size=0.2, random_state=42, stratify=titanic['Survived']
)

print(f'Entrenamiento: {train.shape[0]} filas')
print(f'Prueba:        {test.shape[0]} filas')

# A partir de aquí, cualquier estadístico (media, mediana, moda, cortes de
# discretización) se calcula SOLO con `train` y se aplica igual a `test`.

---
## 4. Eliminar o imputar: decisiones justificadas

Aplicamos las tres decisiones de la diapositiva 3 a nuestras columnas con faltantes.

In [ ]:
# 1) Excluir variable: 'Cabin' tiene demasiados faltantes (~77%)
train = train.drop(columns=['Cabin']).copy()
test = test.drop(columns=['Cabin']).copy()
print("'Cabin' excluida de train y test.")

In [ ]:
# 2) Imputar 'Age' — pero antes, veamos por qué usamos la MEDIANA y no la media
#    (mismo argumento de la diapositiva 4: la media es sensible a valores extremos)
media_age = train['Age'].mean()
mediana_age = train['Age'].median()
edad_maxima = train['Age'].max()

print(f'Media de la edad (train):    {media_age:.2f} años')
print(f'Mediana de la edad (train):  {mediana_age:.2f} años')
print(f'Edad máxima observada:       {edad_maxima:.0f} años')
print('\nHay pasajeros muy mayores (outliers) que desplazan la media hacia arriba;')
print('la mediana es más robusta frente a esos valores extremos, así que la usamos para imputar.')

train['Age'] = train['Age'].fillna(mediana_age)
test['Age'] = test['Age'].fillna(mediana_age)  # OJO: usamos la mediana de TRAIN, no la de test

In [ ]:
# 3) Imputar 'Embarked' con la moda (variable categórica → diapositiva 4)
moda_embarked = train['Embarked'].mode()[0]
print(f"Moda de 'Embarked' (train): '{moda_embarked}'")

train['Embarked'] = train['Embarked'].fillna(moda_embarked)
test['Embarked'] = test['Embarked'].fillna(moda_embarked)

print('\nFaltantes restantes en train:', train.isnull().sum().sum())
print('Faltantes restantes en test: ', test.isnull().sum().sum())

---
## 5. Discretización de la edad

Vamos a agrupar `Age` en intervalos, probando las tres estrategias de las diapositivas 6, 7 y 8. **Los cortes se aprenden solo con `train`** y se aplican igual a `test` (misma lógica de la sección 3).

### 5.1 Ancho uniforme (diapositiva 6)

Dividimos el rango de `Age` en `k` intervalos de igual amplitud.

In [ ]:
k = 4
bordes_ancho = np.linspace(train['Age'].min(), train['Age'].max(), k + 1)

train['Age_ancho_uniforme'] = pd.cut(train['Age'], bins=bordes_ancho, include_lowest=True)
test['Age_ancho_uniforme'] = pd.cut(test['Age'], bins=bordes_ancho, include_lowest=True)

print('Bordes de los intervalos:', np.round(bordes_ancho, 1))
print('\nCantidad de pasajeros por intervalo (train):')
print(train['Age_ancho_uniforme'].value_counts().sort_index())

### 5.2 Frecuencia uniforme (diapositiva 7)

Ahora formamos grupos con **aproximadamente la misma cantidad de pasajeros**, aunque las amplitudes de los intervalos sean distintas.

In [ ]:
train['Age_frecuencia_uniforme'], bordes_frecuencia = pd.qcut(
    train['Age'], q=k, retbins=True, duplicates='drop'
)
test['Age_frecuencia_uniforme'] = pd.cut(test['Age'], bins=bordes_frecuencia, include_lowest=True)

print('Bordes de los intervalos:', np.round(bordes_frecuencia, 1))
print('\nCantidad de pasajeros por intervalo (train) — deberían ser parecidas entre sí:')
print(train['Age_frecuencia_uniforme'].value_counts().sort_index())

### 5.3 Discretización supervisada: entropía (diapositiva 8)

Esta vez usamos la variable objetivo `Survived` para elegir el corte que **mejor separe a los sobrevivientes de los que no sobrevivieron**, calculando la entropía de cada grupo tal como en la fórmula de la diapositiva:

$$H(S) = -\sum_i p_i \log_2(p_i)$$

Un grupo **puro** (todos sobrevivieron o todos no) tiene entropía 0. Probamos varios cortes candidatos y nos quedamos con el que produce la menor entropía ponderada (es decir, grupos más puros).

In [ ]:
def entropia(etiquetas):
    """Entropía de Shannon de un conjunto de etiquetas (0 = grupo puro)."""
    etiquetas = np.asarray(etiquetas)
    if len(etiquetas) == 0:
        return 0.0
    _, conteos = np.unique(etiquetas, return_counts=True)
    probabilidades = conteos / len(etiquetas)
    return -np.sum(probabilidades * np.log2(probabilidades))

# Probamos como cortes candidatos los percentiles 5, 10, ..., 95 de la edad (solo en train)
candidatos = np.percentile(train['Age'], np.arange(5, 100, 5))

mejor_umbral, mejor_entropia_ponderada = None, np.inf
for umbral in candidatos:
    grupo_izq = train.loc[train['Age'] <= umbral, 'Survived']
    grupo_der = train.loc[train['Age'] > umbral, 'Survived']
    if len(grupo_izq) == 0 or len(grupo_der) == 0:
        continue
    entropia_ponderada = (
        len(grupo_izq) * entropia(grupo_izq) + len(grupo_der) * entropia(grupo_der)
    ) / len(train)
    if entropia_ponderada < mejor_entropia_ponderada:
        mejor_entropia_ponderada = entropia_ponderada
        mejor_umbral = umbral

print(f'Mejor corte encontrado: {mejor_umbral:.1f} años')
print(f'Entropía ponderada resultante: {mejor_entropia_ponderada:.4f}')
print(f"Tasa de supervivencia con Age <= {mejor_umbral:.1f}: "
      f"{train.loc[train['Age'] <= mejor_umbral, 'Survived'].mean():.1%}")
print(f"Tasa de supervivencia con Age >  {mejor_umbral:.1f}: "
      f"{train.loc[train['Age'] > mejor_umbral, 'Survived'].mean():.1%}")

In [ ]:
# Verificación: un árbol de decisión de profundidad 1 (un solo corte) entrenado
# con criterio 'entropy' busca exactamente este mismo tipo de corte óptimo.
arbol_un_corte = DecisionTreeClassifier(max_depth=1, criterion='entropy', random_state=42)
arbol_un_corte.fit(train[['Age']], train['Survived'])

umbral_sklearn = arbol_un_corte.tree_.threshold[0]
print(f'Umbral encontrado por scikit-learn: {umbral_sklearn:.1f} años')
print('(similar al que encontramos "a mano" arriba — ambos coinciden con el famoso')
print(' patrón histórico de "mujeres y niños primero": los pasajeros más jóvenes')
print(' tuvieron una tasa de supervivencia notablemente más alta)')

---
## 6. Codificación de variables categóricas

Como en la diapositiva 9, aplicamos el tipo de codificación adecuado según cada variable.

In [ ]:
# 'Sex' es binaria → una sola columna 0/1 es suficiente (no hace falta one-hot)
train['Sex_bin'] = (train['Sex'] == 'female').astype(int)
test['Sex_bin'] = (test['Sex'] == 'female').astype(int)
print(train[['Sex', 'Sex_bin']].drop_duplicates())

In [ ]:
# 'Embarked' es NOMINAL (C = Cherburgo, Q = Queenstown, S = Southampton — sin orden)
# → one-hot (una columna por categoría) o dummy (una menos, para evitar multicolinealidad)
embarked_onehot = pd.get_dummies(train['Embarked'], prefix='Embarked')
embarked_dummy = pd.get_dummies(train['Embarked'], prefix='Embarked', drop_first=True)

print('One-hot: ', list(embarked_onehot.columns), '→', embarked_onehot.shape[1], 'columnas')
print('Dummy:   ', list(embarked_dummy.columns), '→', embarked_dummy.shape[1], 'columna(s) menos')

train = pd.concat([train, embarked_dummy], axis=1)
# A 'test' le aplicamos la MISMA transformación aprendida en train (mismas columnas resultantes)
test = pd.concat([test, pd.get_dummies(test['Embarked'], prefix='Embarked', drop_first=True)], axis=1)
test = test.reindex(columns=train.columns, fill_value=0)  # por si a test le faltara alguna categoría

train[['Embarked'] + list(embarked_dummy.columns)].drop_duplicates()

**`Pclass`** (1ª, 2ª o 3ª clase) ya viene como un número **ordinal** (1 < 2 < 3 en orden de categoría, aunque no linealmente en precio o comodidad) — no necesita ninguna transformación adicional, a diferencia de `Embarked` que no tiene ningún orden natural entre sus categorías.

---
## 7. Para pensar y discutir en clase

Con la misma estructura de la actividad integradora de las diapositivas (**Problema detectado → Tratamiento → Justificación → Comprobación**), analiza ahora la columna **`Fare`** (tarifa pagada por el pasajero):

1. ¿`Fare` tiene datos faltantes en este dataset? Compruébalo con código.
2. ¿Cómo es su distribución — se parece más a la media o a la mediana? (pista: revisa si hay pasajeros que pagaron tarifas mucho más altas que el resto).
3. Si tuvieras que discretizarla, ¿elegirías ancho uniforme, frecuencia uniforme o discretización supervisada? Justifica tu respuesta.
4. ¿Qué tipo de codificación necesitaría `Fare`? (pista: ya es numérica — ¿realmente necesita codificación?)

No hay una única respuesta "correcta": lo importante es justificar la decisión con evidencia del propio dataset, igual que se hizo en las secciones anteriores.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`2.1_Preprocesamiento_transformacion_datos.pdf`](2.1_Preprocesamiento_transformacion_datos.pdf), usando el dataset del **Titanic**:

- Cómo **diagnosticar** datos faltantes y decidir si eliminar filas, excluir variables o imputar.
- Por qué la **mediana** es más robusta que la **media** frente a valores extremos al imputar.
- Cómo **dividir train/test antes de imputar** para evitar la fuga de información.
- Tres formas de **discretizar** una variable numérica: ancho uniforme, frecuencia uniforme y de forma supervisada usando la entropía.
- Cómo **codificar** variables categóricas según su naturaleza (binaria, nominal, ordinal).

### Recursos adicionales
- [Titanic Dataset en Kaggle](https://www.kaggle.com/datasets/yasserh/titanic-dataset)
- [Documentación de `sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [Documentación de `sklearn.tree.DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [Documentación de `pandas.qcut`](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html)